# EDA ⑧ 정체 구간별 8분 초과율 (RQ2 근거)

구급 코호트(2024, `work/seoul_2024_clean.csv`)와 `work/speed/speed_by_gu_hour.parquet`을 **구 + 신고 시각 직전 완결 시간**으로 결합해, 정체(속도) 구간별 8분 초과율을 확인한다.

**누출 방지**: 신고 시각이 속한 시간대(진행 중)가 아니라, 그 직전에 이미 끝난 시간대의 속도를 쓴다. 예) 14:35 신고 → 13:00~14:00 평균 속도 사용. `hour=0`(00:00~00:59)이면 전날 23:00~24:00 값을 쓴다.

**주의**: 여기서 쓰는 속도 5분위는 단순 EDA용 상대 구간이다. 실제 모델에 쓸 `cong_idx`는 **학습 기간(2019~2022)만으로 계산한 평소 속도(`speed_norm`)** 대비 비율이어야 하며, 이는 모델링 설계 단계에서 별도로 만든다 (지금 여기서 만들지 않음).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path().resolve()
while not (ROOT / "CLAUDE.md").exists():
    ROOT = ROOT.parent

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
ems = pd.read_csv(
    ROOT / "work" / "seoul_2024_clean.csv",
    dtype={"dclr_ymd": str},
    usecols=["dclr_ymd", "hour", "sgg_nm", "late"],
)
ems["dclr_ymd"] = ems["dclr_ymd"].astype(int)
ems["hour"] = ems["hour"].astype(int)

# 직전 완결 시간: hour=0이면 전날 24시 코드, 아니면 당일 hour 코드 그대로
# (speed_by_gu_hour의 hour=h는 [h-1시, h시) 구간 평균이라 EMS hour와 코드가 그대로 대응)
d = pd.to_datetime(ems["dclr_ymd"], format="%Y%m%d")
prev_day = (d - pd.Timedelta(days=1)).dt.strftime("%Y%m%d").astype(int)
ems["join_ymd"] = ems["dclr_ymd"].where(ems["hour"] != 0, prev_day)
ems["join_hour"] = ems["hour"].where(ems["hour"] != 0, 24)

print(f"코호트: {len(ems):,}건")

In [ ]:
speed = pd.read_parquet(ROOT / "work" / "speed" / "speed_by_gu_hour.parquet")
speed_2024 = speed[speed["일자"].between(20240101, 20241231)].rename(
    columns={"일자": "join_ymd", "hour": "join_hour"}
)

merged = ems.merge(
    speed_2024[["join_ymd", "sgg_nm", "join_hour", "avg_speed_kmh"]],
    on=["join_ymd", "sgg_nm", "join_hour"],
    how="left",
)

missing_rate = merged["avg_speed_kmh"].isna().mean()
print(f"결합 실패(속도 결측): {merged['avg_speed_kmh'].isna().sum():,}건 ({missing_rate:.2%})")

In [ ]:
valid = merged.dropna(subset=["avg_speed_kmh"]).copy()
valid["speed_q"] = pd.qcut(
    valid["avg_speed_kmh"], 5,
    labels=["Q1(가장 정체)", "Q2", "Q3", "Q4", "Q5(가장 원활)"],
)

summary = valid.groupby("speed_q", observed=True).agg(
    건수=("late", "size"), 초과율=("late", "mean"),
    속도_min=("avg_speed_kmh", "min"), 속도_max=("avg_speed_kmh", "max"),
)
summary["초과율"] = (summary["초과율"] * 100).round(1)
print(summary)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(summary.index.astype(str), summary["초과율"], color="#4C72B0")
for i, v in enumerate(summary["초과율"]):
    ax.text(i, v + 0.5, f"{v}%", ha="center")
ax.set_ylabel("8분 초과율 (%)")
ax.set_xlabel("속도(정체) 5분위 — 신고 직전 완결 시간, 구 단위")
ax.set_title("정체 구간별 8분 초과율 (서울 2024)")
ax.set_ylim(0, max(summary["초과율"]) + 5)
fig.tight_layout()

out_path = ROOT / "eda" / "11_speed_cong.png"
fig.savefig(out_path, dpi=150)
print("저장:", out_path)